# Binary replication: Llama-3 70B opinion dynamics

Repeats the binary experiment (N = 50 agents, 20 runs, 10 sweeps, opinions k/z, "friends" prompt). Prompt, names, shuffle and update step after De Marzo's `LLMs-Opinion-Dynamics` code, adapted to vLLM batching. Writes `data/plot_sources/binary_replication.csv`.

In [ ]:
import random
import re
import string
from pathlib import Path

import pandas as pd
from vllm import LLM, SamplingParams

In [ ]:
SEED = 7
MODEL_ID = "TechxGenus/Meta-Llama-3-70B-Instruct-AWQ"

N = 50
N_RUNS = 20
T_MAX = 10 * N          # 10 sweeps
OPINIONS = ["k", "z"]
NAME_LENGTH = 3

TEMPERATURE = 0.2
TOP_P = 0.9
MAX_NEW_TOKENS = 16

OUT = Path("../data/plot_sources/binary_replication.csv")

In [ ]:
def random_names(n, rng):
    chars = string.ascii_letters + string.digits
    names = set()
    while len(names) < n:
        names.add("".join(rng.choices(chars, k=NAME_LENGTH)))
    return list(names)


def create_prompt(names, opinions):
    # wording of the original study
    text = (
        "Below you can see the list of all your friends together with the opinion they support.\n"
        "        You must reply with the opinion you want to support.\n"
        "        The opinion must be reported between square brackets.\n"
    )
    for name, opinion in zip(names, opinions):
        text += f"{name}: {opinion}\n"
    text += ". \n        Reply only with the opinion you want to support, between square brackets."
    return text


def parse_reply(text):
    # label between square brackets
    found = re.findall(r"\[([^\]]+)\]", text)
    for c in found if found else [text]:
        c = c.strip().strip(" .,:;!?'\"")
        if c in OPINIONS:
            return c
    return None


def chat_format(llm, prompts):
    tok = llm.get_tokenizer()
    return [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
            for p in prompts]

In [ ]:
def magnetization(run):
    return (2 * run["Na"] - N) / N


def build_prompt(run):
    rng = run["rng"]
    # labels swapped with probability 1/2
    labels = OPINIONS[:] if rng.random() <= 0.5 else OPINIONS[::-1]
    to_display = dict(zip(OPINIONS, labels))
    to_internal = dict(zip(labels, OPINIONS))
    names = random_names(N, rng)
    opinions = [OPINIONS[0]] * run["Na"] + [OPINIONS[1]] * (N - run["Na"])
    pairs = list(zip(names, opinions))
    rng.shuffle(pairs)
    i = rng.randint(0, N - 1)                # focal agent, not shown in the list
    others = [(n, to_display[o]) for k, (n, o) in enumerate(pairs) if k != i]
    prompt = create_prompt([n for n, _ in others], [o for _, o in others])
    return prompt, pairs[i][1], to_internal


def record(run, step):
    m = magnetization(run)
    return {"run": run["run"], "step": step, "time": step / N, "magnetization": m,
            "coordination_level": abs(m), "invalid_count": run["invalid"]}

In [ ]:
llm = LLM(model=MODEL_ID, quantization="awq", trust_remote_code=True, tensor_parallel_size=1,
          gpu_memory_utilization=0.90, max_model_len=2048, max_num_seqs=N_RUNS, enforce_eager=True)
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS)

In [ ]:
runs = [{"run": r, "Na": N // 2, "invalid": 0, "rng": random.Random(SEED + r)} for r in range(N_RUNS)]
rows = [record(run, 0) for run in runs]

for step in range(1, T_MAX + 1):
    active = [run for run in runs if abs(magnetization(run)) < 1]      # consensus is absorbing
    prompts = [build_prompt(run) for run in active]
    outputs = llm.generate(chat_format(llm, [p[0] for p in prompts]), sampling, use_tqdm=False) if active else []
    for run, (_, own, to_internal), out in zip(active, prompts, outputs):
        chosen = parse_reply(out.outputs[0].text)
        if chosen is None:
            run["invalid"] += 1
        elif to_internal[chosen] != own:
            run["Na"] += 1 if to_internal[chosen] == OPINIONS[0] else -1
    rows += [record(run, step) for run in runs]
    if step % N == 0:
        print(f"t={step // N}  mean C={sum(abs(magnetization(r)) for r in runs) / N_RUNS:.3f}")

df = pd.DataFrame(rows)
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print(df.groupby("run").tail(1)[["run", "coordination_level"]].to_string(index=False))